# 04 - Champion Selection & Experiment Balance Check

This notebook:
1. Queries `data/db.sqlite3` for completed LLaMEA experiments across target BBOB problems (`f1, f8, f11, f15, f21`) at `dim = 5` and `noise_std = 0.05`.
2. Verifies the **experiment balance table** (equal completed runs per problem).
3. Selects the **problem-specific champion algorithm** (lowest `final_error` across all iterations of all completed experiments for that problem).
4. Exports `data/champions.json` for evaluation in Notebook 05.

In [ ]:
import json
import sqlite3
import pandas as pd
from pathlib import Path

# Paths
PROJECT_ROOT = Path('../').resolve()
DB_PATH = PROJECT_ROOT / 'data' / 'db.sqlite3'
CHAMPIONS_PATH = PROJECT_ROOT / 'data' / 'champions.json'

TARGET_PROBLEMS = [1, 8, 11, 15, 21]
TARGET_NOISE_STD = 0.05
TARGET_DIM = 5

print(f'Database Path: {DB_PATH}')
print(f'Target Problems: {TARGET_PROBLEMS}')
print(f'Target Noise std: {TARGET_NOISE_STD}')
print(f'Target Dim: {TARGET_DIM}')

## 1. Experiment Balance Check

In [ ]:
conn = sqlite3.connect(DB_PATH)

# Query completed experiments matching target criteria
query = """
SELECT 
    id as exp_id,
    problem_id,
    dim,
    noise_std,
    prompt_strategy,
    status,
    best_algorithm,
    best_final_error
FROM experiments
WHERE dim = 5 
  AND noise_std = 0.05
  AND status = 'completed'
ORDER BY problem_id, exp_id
"""
df_exps = pd.read_sql_query(query, conn)
print(f'Total completed 5D (noise_std=0.05) experiments: {len(df_exps)}')

# Balance Table
balance_summary = []
for p_id in TARGET_PROBLEMS:
    subset = df_exps[df_exps['problem_id'] == p_id]
    count = len(subset)
    balance_summary.append({
        'problem_id': f'f{p_id}',
        'completed_experiments': count,
        'status': 'OK' if count > 0 else 'NO_DATA'
    })

df_balance = pd.DataFrame(balance_summary)
print('\n=== Experiment Balance Table ===')
print(df_balance.to_string(index=False))

counts = df_balance['completed_experiments'].unique()
if len(counts) == 1 and counts[0] > 0:
    print(f'\nSUCCESS: All target problems are perfectly balanced with {counts[0]} completed experiment(s) each.')
else:
    print('\nWARNING: Experiments are unbalanced or missing. Run additional LLaMEA sessions before final evaluation.')

## 2. Select Problem-Specific Champions

In [ ]:
# Query all iterations from completed experiments to find true absolute lowest final_error per problem
iter_query = """
SELECT 
    e.problem_id,
    e.id as experiment_id,
    e.llm_name,
    e.prompt_strategy,
    i.id as iteration_id,
    i.algorithm_name,
    i.final_error,
    i.evaluations_used,
    i.code_path
FROM iterations i
JOIN experiments e ON i.experiment_id = e.id
WHERE e.dim = 5
  AND e.noise_std = 0.05
  AND e.status = 'completed'
  AND i.final_error IS NOT NULL
ORDER BY i.final_error ASC
"""
df_iters = pd.read_sql_query(iter_query, conn)
conn.close()

champions = {}

print('=== Problem-Specific Champions ===')
for p_id in TARGET_PROBLEMS:
    p_subset = df_iters[df_iters['problem_id'] == p_id]
    if len(p_subset) == 0:
        print(f'f{p_id}: No completed iterations found.')
        continue
    best_row = p_subset.iloc[0]
    
    champions[str(p_id)] = {
        'problem_id': p_id,
        'experiment_id': int(best_row['experiment_id']),
        'algorithm_name': str(best_row['algorithm_name']),
        'final_error': float(best_row['final_error']),
        'evaluations_used': int(best_row['evaluations_used']) if pd.notnull(best_row['evaluations_used']) else None,
        'code_path': str(best_row['code_path']),
        'llm_name': str(best_row['llm_name']),
        'prompt_strategy': str(best_row['prompt_strategy'])
    }
    
    print(f"f{p_id}: {best_row['algorithm_name']} (Exp #{best_row['experiment_id']}) -> final_error = {best_row['final_error']:.6e}")

# Save champions.json
CHAMPIONS_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(CHAMPIONS_PATH, 'w') as f:
    json.dump(champions, f, indent=2)

print(f'\nExported champions to {CHAMPIONS_PATH}')